# Baseline logistic regression + calibration (synthetic demo)

**Purpose:** Demonstrate the evaluation pipeline (split → baseline model → Brier / calibration) on synthetic data only.

**Not clinical evidence.** See `docs/protocol.md`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if (ROOT / "src").exists():
    sys.path.insert(0, str(ROOT / "src"))
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / "src"))

from bbc.metrics import brier_score, calibration_slope_intercept

DATA = ROOT / "data" / "synthetic" / "synthetic_cohort.csv"
df = pd.read_csv(DATA)
df.head()

In [ ]:
features = ["age", "screening_gap_years", "density_score"]
X = df[features]
y = df["event"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=20260812, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
p_test = model.predict_proba(X_test)[:, 1]

metrics = {
    "brier": brier_score(y_test.to_numpy(), p_test),
    "auc": roc_auc_score(y_test, p_test),
}
slope, intercept = calibration_slope_intercept(y_test.to_numpy(), p_test)
metrics["calibration_slope"] = slope
metrics["calibration_intercept"] = intercept
metrics

In [ ]:
# Simple reliability-style binning plot
bins = np.linspace(0, 1, 6)
dig = np.digitize(p_test, bins) - 1
bin_center, emp_rate = [], []
for b in range(len(bins) - 1):
    mask = dig == b
    if mask.sum() == 0:
        continue
    bin_center.append(p_test[mask].mean())
    emp_rate.append(y_test.to_numpy()[mask].mean())

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
ax.plot(bin_center, emp_rate, "o-", label="logistic baseline")
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed event rate")
ax.set_title("Calibration plot (synthetic demo)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Next steps

1. Lock a lawful public dataset in `docs/data.md`.
2. Freeze protocol amendments in git.
3. Add a Bayesian GLM notebook only after this baseline path is stable.